# Day 2 — Re-evaluate the fine-tuned checkpoints (Kaggle T4)

Run AFTER `kaggle_train_illum.ipynb` finishes and you've uploaded `best_baseline_ft.pth` + `best_illum_ft.pth` as a Kaggle dataset (`lumidiff-day2-ckpts`).

Produces:
1. **Headline table for the +illum (fine-tuned) model** — eval15 + LOL-v2 Real, 5 + 20 DDIM steps, with LPIPS.
2. **Method ablation row** — baseline_ft vs illum_ft (both fine-tuned identically except for the prior flag).
3. **Step ablation for illum_ft** — 5/10/20/50/100 steps to confirm the few-step convergence story holds.
4. **Adaptive Residual Rescaling (ARR) grid** on illum_ft.
5. **T4 efficiency** for the new model.
6. **Updated Figures 1 + 3.**

Wall time: ~45–75 min.

In [ ]:
# === Cell 1: install ===
!pip install -q --upgrade pip
!pip install -q scikit-image lpips fvcore matplotlib

In [ ]:
# === Cell 2: clone repo ===
import os, sys, shutil, subprocess
BRANCH = "main"
REPO_URL = "https://github.com/chirana07/Diffusion_new_final.git"
REPO_DIR = "/kaggle/working/Diffunet"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

for required in ("kaggle_path_discovery.py", "make_teaser_figure.py", "make_step_ablation_figure.py"):
    if not os.path.exists(required):
        raise SystemExit(f"{required} missing from cloned repo. Push your latest commits.")
print("Repo ready.")

In [ ]:
# === Cell 3: PREFLIGHT — discover datasets ===
import kaggle_path_discovery as kpd
kpd.diagnose("/kaggle/input")
discoveries = kpd.discover_all("/kaggle/input")
kpd.print_picks(discoveries)

# === MANUAL OVERRIDE if discovery picked the wrong dirs ===
# discoveries['lolv2_real_test'] = {'low_dir': '...', 'high_dir': '...', 'n_images': 100, 'kind': 'test'}
# discoveries['eval15_test'] = {'low_dir': '...', 'high_dir': '...', 'n_images': 15, 'kind': 'test'}

if not discoveries.get('lolv2_real_test'):
    raise SystemExit("LOL-v2 Real Test split not found. Re-attach via right panel + Add Data, or set the path manually above.")
if not discoveries.get('eval15_test'):
    print("NOTE: eval15 not found. The eval15 rows will be skipped.")

EVAL15_LOW  = discoveries['eval15_test']['low_dir']  if discoveries.get('eval15_test') else None
EVAL15_HIGH = discoveries['eval15_test']['high_dir'] if discoveries.get('eval15_test') else None
LOLV2_LOW   = discoveries['lolv2_real_test']['low_dir']
LOLV2_HIGH  = discoveries['lolv2_real_test']['high_dir']

In [ ]:
# === Cell 4: locate the fine-tuned checkpoints ===
import glob
all_ckpts = sorted(glob.glob("/kaggle/input/**/*.pth", recursive=True))
ILLUM_CKPT    = next((c for c in all_ckpts if "illum_ft"    in c.lower() and "best" in c.lower()), None)
BASELINE_CKPT = next((c for c in all_ckpts if "baseline_ft" in c.lower() and "best" in c.lower()), None)

# === MANUAL OVERRIDE ===
# ILLUM_CKPT    = "/kaggle/input/lumidiff-day2-ckpts/best_illum_ft.pth"
# BASELINE_CKPT = "/kaggle/input/lumidiff-day2-ckpts/best_baseline_ft.pth"

if ILLUM_CKPT is None:
    raise SystemExit(
        "best_illum_ft.pth not found under /kaggle/input/.\n"
        "Did you upload it as a Kaggle dataset? Datasets -> + New Dataset -> upload the .pth files,\n"
        "name it 'lumidiff-day2-ckpts', then attach via right panel -> + Add Data."
    )
print(f"ILLUM_CKPT    = {ILLUM_CKPT}")
print(f"BASELINE_CKPT = {BASELINE_CKPT}  ({'OK' if BASELINE_CKPT else 'missing - method ablation will be skipped'})")

EVAL_RESULTS = "/kaggle/working/eval_results_day2"
os.makedirs(EVAL_RESULTS, exist_ok=True)

SPLITS = []
if EVAL15_LOW:
    SPLITS.append(f"eval15:{EVAL15_LOW}:{EVAL15_HIGH}")
SPLITS.append(f"lolv2_real:{LOLV2_LOW}:{LOLV2_HIGH}")
print("Splits:", SPLITS)

In [ ]:
# === Cell 5: GPU sanity ===
import torch
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
if not torch.cuda.is_available():
    raise SystemExit("Set Accelerator -> GPU T4 -> Save, then re-run.")

In [ ]:
# === Cell 6: HEADLINE — illum_ft at 5 and 20 DDIM steps with LPIPS ===
import csv, sys, subprocess
for STEPS in (5, 20):
    cmd = [
        sys.executable, "evaluation.py",
        "--splits", *SPLITS,
        "--inference-steps", str(STEPS),
        "--sampler", "ddim",
        "--checkpoint", ILLUM_CKPT,
        "--results-root", os.path.join(EVAL_RESULTS, "headline_illum"),
        "--tag", f"s{STEPS}",
    ]
    print("$ " + " ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# === Cell 7: METHOD ABLATION — baseline_ft @ 5 steps (only if checkpoint exists) ===
if BASELINE_CKPT is not None:
    cmd = [
        sys.executable, "evaluation.py",
        "--splits", *SPLITS,
        "--inference-steps", "5",
        "--sampler", "ddim",
        "--checkpoint", BASELINE_CKPT,
        "--results-root", os.path.join(EVAL_RESULTS, "method_ablation"),
        "--tag", "baseline_s5",
    ]
    print("$ " + " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skipping method ablation: no baseline_ft checkpoint.")

In [ ]:
# === Cell 8: STEP ABLATION on illum_ft — confirms 5-step claim survives ===
for N in (5, 10, 20, 50, 100):
    cmd = [
        sys.executable, "evaluation.py",
        "--splits", *SPLITS,
        "--inference-steps", str(N),
        "--sampler", "ddim",
        "--checkpoint", ILLUM_CKPT,
        "--results-root", os.path.join(EVAL_RESULTS, "step_ablation_full"),
        "--tag", f"ddim_s{N}",
        "--no-lpips",
    ]
    print("$ " + " ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# === Cell 9: ARR grid on illum_ft @ 5 steps ===
ARR_GRID = []
for ALPHA in (0.0, 0.1, 0.2, 0.3, 0.4, 0.5):
    tag = f"arr_a{int(ALPHA*100):03d}"
    cmd = [
        sys.executable, "evaluation.py",
        "--splits", f"lolv2_real:{LOLV2_LOW}:{LOLV2_HIGH}",
        "--inference-steps", "5",
        "--sampler", "ddim",
        "--checkpoint", ILLUM_CKPT,
        "--results-root", os.path.join(EVAL_RESULTS, "arr_grid"),
        "--tag", tag,
        "--gate-alpha", str(ALPHA),
        "--gate-floor", "0.5",
        "--no-lpips",
    ]
    print("$ " + " ".join(cmd))
    subprocess.run(cmd, check=True)
    summary_path = os.path.join(EVAL_RESULTS, f"arr_grid_{tag}", "summary.csv")
    with open(summary_path) as f:
        for row in csv.DictReader(f):
            row["alpha"] = ALPHA
            ARR_GRID.append(row)

In [ ]:
# === Cell 10: T4 efficiency for illum_ft ===
cmd = [
    sys.executable, "measure_efficiency.py",
    "--steps", "5", "10", "20",
    "--resolution", "400", "600",
    "--device", "cuda",
    "--checkpoint", ILLUM_CKPT,
    "--out-csv", os.path.join(EVAL_RESULTS, "efficiency_t4_illum.csv"),
]
print("$ " + " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# === Cell 11: Render Figure 1 (teaser) and Figure 3 (step ablation) ===
subprocess.run([
    sys.executable, "make_teaser_figure.py",
    "--pred-s5",  os.path.join(EVAL_RESULTS, "headline_illum_s5/lolv2_real"),
    "--pred-s20", os.path.join(EVAL_RESULTS, "headline_illum_s20/lolv2_real"),
    "--low",      LOLV2_LOW,
    "--gt",       LOLV2_HIGH,
    "--per-image-csv",     os.path.join(EVAL_RESULTS, "headline_illum_s5/per_image.csv"),
    "--per-image-csv-s20", os.path.join(EVAL_RESULTS, "headline_illum_s20/per_image.csv"),
    "--split", "lolv2_real",
    "--out",     os.path.join(EVAL_RESULTS, "figure1_teaser_illum.pdf"),
    "--out-png", os.path.join(EVAL_RESULTS, "figure1_teaser_illum.png"),
], check=True)
subprocess.run([
    sys.executable, "make_step_ablation_figure.py",
    "--eval-root", EVAL_RESULTS,
    "--out",     os.path.join(EVAL_RESULTS, "figure3_step_ablation_illum.pdf"),
    "--out-png", os.path.join(EVAL_RESULTS, "figure3_step_ablation_illum.png"),
], check=True)

In [ ]:
# === Cell 12: print the four paper tables ===
def read_summary(path):
    if not os.path.exists(path): return []
    return list(csv.DictReader(open(path)))

def fmt(r, variant):
    lp = r.get('lpips_mean')
    lp_s = f"{float(lp):.4f}" if lp not in (None, '', 'None') else '-'
    return (f"| {variant} | {r['split']} | {r['n']} | {r['inference_steps']} | "
            f"{float(r['psnr_mean']):.3f} | {float(r['ssim_mean']):.4f} | {lp_s} | "
            f"{float(r['runtime_mean']):.3f} |")

print("\n### Table A — Headline (illum_ft)\n")
print("| Variant | Split | n | Steps | PSNR | SSIM | LPIPS | Latency/img (s) |")
print("|---|---|---|---|---|---|---|---|")
for tag in ("s5", "s20"):
    for r in read_summary(os.path.join(EVAL_RESULTS, f"headline_illum_{tag}/summary.csv")):
        print(fmt(r, "+illum (FT)"))

print("\n### Table B — Method ablation (5 DDIM steps, both fine-tuned identically)\n")
print("| Variant | Split | n | Steps | PSNR | SSIM | LPIPS | Latency/img (s) |")
print("|---|---|---|---|---|---|---|---|")
for r in read_summary(os.path.join(EVAL_RESULTS, "method_ablation_baseline_s5/summary.csv")):
    print(fmt(r, "baseline_ft (no prior)"))
for r in read_summary(os.path.join(EVAL_RESULTS, "headline_illum_s5/summary.csv")):
    print(fmt(r, "+ illum prior (ours)"))

print("\n### Table C — Step ablation (illum_ft)\n")
print("| Steps | Split | PSNR | SSIM | Latency/img (s) |")
print("|---|---|---|---|---|")
for N in (5, 10, 20, 50, 100):
    for r in read_summary(os.path.join(EVAL_RESULTS, f"step_ablation_full_ddim_s{N}/summary.csv")):
        print(f"| {N} | {r['split']} | {float(r['psnr_mean']):.3f} | {float(r['ssim_mean']):.4f} | {float(r['runtime_mean']):.3f} |")

print("\n### Table D — Adaptive Residual Rescaling (illum_ft, 5 DDIM steps, LOL-v2 Real)\n")
print("| alpha | PSNR | SSIM |")
print("|---|---|---|")
for r in ARR_GRID:
    print(f"| {float(r['alpha']):.2f} | {float(r['psnr_mean']):.3f} | {float(r['ssim_mean']):.4f} |")

print("\n### Efficiency (T4, 400x600, illum_ft)\n")
with open(os.path.join(EVAL_RESULTS, "efficiency_t4_illum.csv")) as f:
    print(f.read())

In [ ]:
# === Cell 13: bundle ===
import shutil
out_zip = "/kaggle/working/phase3_day2_eval_outputs.zip"
if os.path.exists(out_zip): os.remove(out_zip)
shutil.make_archive(out_zip.replace(".zip", ""), "zip", EVAL_RESULTS)
print("Wrote", out_zip)
print("Download via Kaggle right panel -> Output tab.")